In [5]:
import torch
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn

train_dataset= datasets.MNIST(root='.\data', train=True, download=True, transform=ToTensor())
test_dataset= datasets.MNIST(root='.\data', train=False, transform=ToTensor())
train_loader=torch.utils.data.DataLoader(dataset=train_dataset, batch_size=100, shuffle=True)
test_loader=torch.utils.data.DataLoader(dataset=test_dataset, batch_size=100, shuffle=False)

device=( 'cuda' if torch.cuda.is_available() else 'cpu')
print(device)

class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1=nn.Sequential(nn.Conv2d(1, 10, 5), nn.ReLU(), nn.MaxPool2d(kernel_size=2))
    self.conv2=nn.Sequential(nn.Conv2d(10, 20, 5), nn.ReLU(), nn.MaxPool2d(kernel_size=2))
    self.fc1=nn.Linear(320, 10)
    self.fc2=nn.Linear(10, 20)
    self.fc3=nn.Linear(20, 10)
  def forward(self, x):
    x=self.conv1(x)
    x=self.conv2(x)
    x=x.view(x.size(0), -1)
    x=self.fc1(x)
    x=self.fc2(x)
    x=self.fc3(x)
    return x

model=CNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(2):
  for i, (images, labels) in enumerate (train_loader):
    images=images.to(device)
    labels=labels.to(device)
    outputs=model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
     n_correct = 0
     n_samples = 0
     for images, labels in test_loader:
         images = images.to(device)
         labels = labels.to(device)
         outputs = model(images)
         predicted = torch.argmax(outputs.data, 1)
         n_samples += labels.size(0)
         n_correct += (predicted == labels).sum().item()

acc = 100.0 * n_correct / n_samples
print(acc)

cpu
98.25
